# AI Use Case 4 — Dataclass: o runtime interno de um agente

**Problema:** dentro de um agente circula muito dado **interno e confiável**: configuração da chamada, histórico de mensagens,
contagem de tokens e custo, *spans* de rastreamento, casos e resultados de avaliação. Esses objetos são criados aos milhares
(um por mensagem, um por passo, um por caso de teste) e **não vêm de fora** — validá-los com Pydantic a cada criação é custo sem benefício.

**Ideia central:** Pydantic na **borda** (o que vem do LLM, do usuário, da config); dataclasses no **núcleo** (estado e
contabilidade do agente). Imutáveis quando representam um fato (`frozen`), com `slots` quando são muitos.

```
  ┌──────────── borda (Pydantic) ────────────┐      ┌──────────────── núcleo (dataclass) ────────────────┐
  │ settings, saída do LLM, args de tools,   │ ───► │ LLMConfig · Message · Conversation · Usage · Span  │
  │ request/response da API                  │      │ EvalCase · EvalResult · EvalReport                 │
  └──────────────────────────────────────────┘      └────────────────────────────────────────────────────┘
```

Roteiro:
1. `LLMConfig` imutável e variantes com `replace`
2. Mensagens e memória com orçamento de tokens
3. Contabilidade de uso e custo (`__add__`)
4. Rastreamento (*tracing*) com spans
5. Harness de avaliação (evals)
6. Por que dataclass aqui (e não Pydantic)

In [1]:
import time, json, statistics
from collections import Counter
from contextlib import contextmanager
from dataclasses import dataclass, field, replace, asdict, fields
from enum import StrEnum
from typing import Any, Literal
from fake_llm import ScriptedLLM, keyword_llm

## 1. `LLMConfig` imutável

Configuração de chamada é um **valor**: não deve mudar no meio de uma execução (bug clássico: uma função altera `temperature`
de um objeto compartilhado). `frozen=True` + `replace()` dá variantes seguras — ótimo para experimentos e testes A/B.

In [2]:
@dataclass(frozen=True, slots=True, kw_only=True)
class LLMConfig:
    model: str
    max_tokens: int = 1024
    temperature: float = 0.0
    top_p: float | None = None
    stop_sequences: tuple[str, ...] = ()
    system: str = ""

    def __post_init__(self):
        if not 0 <= self.temperature <= 1:
            raise ValueError("temperature deve estar em [0, 1]")
        if self.max_tokens <= 0:
            raise ValueError("max_tokens deve ser positivo")

    def to_request(self) -> dict[str, Any]:
        """Só os campos definidos, no formato de kwargs do SDK."""
        req = {f.name: getattr(self, f.name) for f in fields(self)}
        return {k: (list(v) if isinstance(v, tuple) else v) for k, v in req.items() if v not in (None, (), "")}

base = LLMConfig(model="medium-model", system="Você é um assistente de suporte.")
creative = replace(base, temperature=0.8, max_tokens=2048)
sweep = [replace(base, temperature=t) for t in (0.0, 0.3, 0.7)]

print(f"base.to_request(): {base.to_request()}")
print(f"creative: {creative}")
print(f"[c.temperature for c in sweep]: {[c.temperature for c in sweep]}")
print(f"base intacta: {base.temperature}")
cache = {base: "resposta em cache"}
same_config = LLMConfig(model="medium-model", system="Você é um assistente de suporte.")
print(f"cache[same_config] (hashable -> chave de cache): {cache[same_config]}")
try:
    base.temperature = 1.0
except Exception as e:
    print(f"{type(e).__name__}: {e}")

base.to_request(): {'model': 'medium-model', 'max_tokens': 1024, 'temperature': 0.0, 'system': 'Você é um assistente de suporte.'}
creative: LLMConfig(model='medium-model', max_tokens=2048, temperature=0.8, top_p=None, stop_sequences=(), system='Você é um assistente de suporte.')
[c.temperature for c in sweep]: [0.0, 0.3, 0.7]
base intacta: 0.0
cache[same_config] (hashable -> chave de cache): resposta em cache
FrozenInstanceError: cannot assign to field 'temperature'


## 2. Mensagens e memória com orçamento de tokens

Conversas longas estouram a janela de contexto (e o custo). Uma estratégia simples: manter o system prompt, as primeiras mensagens
(objetivo da conversa) e as mais recentes que couberem no orçamento.

> A contagem de tokens aqui é uma aproximação (`len/4`). Em produção use o contador do provedor — por exemplo, com o SDK da
> Anthropic (referência, não executado):
>
> ```python
> from anthropic import Anthropic
> client = Anthropic()
>
> def count_tokens(messages: list[Message], system: str, model: str = "claude-opus-5-5") -> int:
>     resp = client.messages.count_tokens(model=model, system=system, messages=[m.to_api() for m in messages])
>     return resp.input_tokens
> ```
>
> e trocar `approx_tokens` por esse contador no `window()` (contando em lote para não fazer uma chamada por mensagem).

In [3]:
class Role(StrEnum):
    USER = "user"
    ASSISTANT = "assistant"

@dataclass(frozen=True, slots=True)
class Message:
    role: Role
    content: str

    @property
    def approx_tokens(self) -> int:
        return max(1, len(self.content) // 4) + 4        # +4: overhead aproximado por mensagem

    def to_api(self) -> dict[str, str]:
        return {"role": self.role.value, "content": self.content}

@dataclass(slots=True)
class Conversation:
    system: str
    messages: list[Message] = field(default_factory=list)
    keep_first: int = 2

    def add(self, role: Role, content: str) -> None:
        self.messages.append(Message(role, content))

    def window(self, budget_tokens: int) -> list[Message]:
        budget = budget_tokens - len(self.system) // 4
        head = self.messages[: self.keep_first]
        budget -= sum(m.approx_tokens for m in head)
        tail: list[Message] = []
        for m in reversed(self.messages[self.keep_first:]):
            if m.approx_tokens > budget:
                break
            tail.append(m)
            budget -= m.approx_tokens
        tail.reverse()
        dropped = len(self.messages) - len(head) - len(tail)
        marker = [Message(Role.USER, f"[{dropped} mensagens antigas omitidas]")] if dropped else []
        return head + marker + tail

conv = Conversation(system="Você é um assistente de viagens.")
conv.add(Role.USER, "Quero planejar 10 dias no Japão em abril, orçamento médio.")
conv.add(Role.ASSISTANT, "Ótimo! Tóquio, Kyoto e Osaka são uma boa base. " * 3)
for i in range(12):
    conv.add(Role.USER, f"Pergunta {i}: e sobre o dia {i + 1}? " + "detalhes " * 15)
    conv.add(Role.ASSISTANT, f"Resposta {i}: sugestão para o dia {i + 1}. " + "explicação " * 20)

win = conv.window(budget_tokens=600)
print(f"total: {len(conv.messages)} mensagens, ~{sum(m.approx_tokens for m in conv.messages)} tokens")
print(f"janela: {len(win)} mensagens, ~{sum(m.approx_tokens for m in win)} tokens")
for m in win:
    print(f"  {m.role:<9} {m.content[:60]}")

total: 26 mensagens, ~1404 tokens
janela: 12 mensagens, ~586 tokens
  user      Quero planejar 10 dias no Japão em abril, orçamento médio.
  assistant Ótimo! Tóquio, Kyoto e Osaka são uma boa base. Ótimo! Tóquio
  user      [15 mensagens antigas omitidas]
  assistant Resposta 7: sugestão para o dia 8. explicação explicação exp
  user      Pergunta 8: e sobre o dia 9? detalhes detalhes detalhes deta
  assistant Resposta 8: sugestão para o dia 9. explicação explicação exp
  user      Pergunta 9: e sobre o dia 10? detalhes detalhes detalhes det
  assistant Resposta 9: sugestão para o dia 10. explicação explicação ex
  user      Pergunta 10: e sobre o dia 11? detalhes detalhes detalhes de
  assistant Resposta 10: sugestão para o dia 11. explicação explicação e
  user      Pergunta 11: e sobre o dia 12? detalhes detalhes detalhes de
  assistant Resposta 11: sugestão para o dia 12. explicação explicação e


## 3. Contabilidade de uso e custo

`Usage` é um valor somável: implementar `__add__` permite `sum(usages, Usage())` sobre todas as chamadas de uma execução.
A tabela de preços é um dicionário simples (ilustrativo).

In [4]:
PRICES_PER_MTOK = {"small-model": (1.0, 5.0), "medium-model": (3.0, 15.0), "large-model": (5.0, 25.0)}   # ilustrativo

@dataclass(frozen=True, slots=True)
class Usage:
    input_tokens: int = 0
    output_tokens: int = 0
    calls: int = 0
    cost_usd: float = 0.0

    @classmethod
    def from_call(cls, model: str, tokens_in: int, tokens_out: int) -> "Usage":
        p_in, p_out = PRICES_PER_MTOK[model]
        return cls(tokens_in, tokens_out, 1, tokens_in / 1e6 * p_in + tokens_out / 1e6 * p_out)

    def __add__(self, other: "Usage") -> "Usage":
        return Usage(self.input_tokens + other.input_tokens, self.output_tokens + other.output_tokens,
                     self.calls + other.calls, self.cost_usd + other.cost_usd)

per_step = [Usage.from_call("small-model", 1_200, 80), Usage.from_call("large-model", 15_000, 1_100),
            Usage.from_call("large-model", 16_500, 600), Usage.from_call("medium-model", 4_000, 300)]
total = sum(per_step, Usage())
print(f"total: {total}")
print(f"{total.calls} chamadas | {total.input_tokens:,} in / {total.output_tokens:,} out | US$ {total.cost_usd:.4f}")
print(f"custo por chamada: {[f'{u.cost_usd:.4f}' for u in per_step]}")

total: Usage(input_tokens=36700, output_tokens=2080, calls=4, cost_usd=0.21810000000000002)
4 chamadas | 36,700 in / 2,080 out | US$ 0.2181
custo por chamada: ['0.0016', '0.1025', '0.0975', '0.0165']


## 4. Tracing com spans

Para depurar agentes você precisa ver **o que aconteceu, em que ordem, quanto tempo levou e quanto custou**. Um `Span` é um registro
leve (milhares por execução → `slots=True`); um context manager mede o tempo e monta a árvore.

In [5]:
@dataclass(slots=True)
class Span:
    name: str
    kind: Literal["agent", "llm", "tool", "step"]
    start: float
    end: float | None = None
    attrs: dict[str, Any] = field(default_factory=dict)
    usage: Usage = field(default_factory=Usage)
    children: list["Span"] = field(default_factory=list)
    error: str | None = None

    @property
    def duration_ms(self) -> float:
        return ((self.end or self.start) - self.start) * 1000

    def total_usage(self) -> Usage:
        return sum((c.total_usage() for c in self.children), self.usage)

class Tracer:
    def __init__(self):
        self.root: Span | None = None
        self._stack: list[Span] = []

    @contextmanager
    def span(self, name: str, kind: str, **attrs):
        s = Span(name, kind, time.perf_counter(), attrs=attrs)
        (self._stack[-1].children if self._stack else []).append(s)
        self.root = self.root or s
        self._stack.append(s)
        try:
            yield s
        except Exception as e:
            s.error = f"{type(e).__name__}: {e}"
            raise
        finally:
            s.end = time.perf_counter()
            self._stack.pop()

    def render(self, s: Span | None = None, depth: int = 0) -> None:
        s = s or self.root
        u = s.total_usage()
        cost = f" ${u.cost_usd:.4f}" if u.calls else ""
        err = f"  ✘ {s.error}" if s.error else ""
        print(f"{'  ' * depth}{'└─ ' if depth else ''}{s.kind}:{s.name} {s.duration_ms:.1f}ms{cost}{err}")
        for c in s.children:
            self.render(c, depth + 1)

tracer = Tracer()
llm = ScriptedLLM(['{"plan": ["buscar pedido", "responder"]}', "Seu pedido chega amanhã."])

with tracer.span("support_agent", "agent", user="ana"):
    with tracer.span("plan", "llm", model="small-model") as s:
        r = llm([{"role": "user", "content": "Cadê meu pedido 123?"}])
        s.usage = Usage.from_call("small-model", r.input_tokens, r.output_tokens)
    with tracer.span("get_order", "tool", order_id="123"):
        time.sleep(0.01)
    try:
        with tracer.span("get_tracking", "tool"):
            raise TimeoutError("transportadora não respondeu")
    except TimeoutError:
        pass
    with tracer.span("answer", "llm", model="large-model") as s:
        r = llm([{"role": "user", "content": "Responda com os dados do pedido"}])
        s.usage = Usage.from_call("large-model", r.input_tokens, r.output_tokens)

tracer.render()
print(f"\nuso total: {tracer.root.total_usage()}")

agent:support_agent 10.2ms $0.0002
  └─ llm:plan 0.0ms $0.0001
  └─ tool:get_order 10.1ms
  └─ tool:get_tracking 0.0ms  ✘ TimeoutError: transportadora não respondeu
  └─ llm:answer 0.0ms $0.0002

uso total: Usage(input_tokens=12, output_tokens=16, calls=2, cost_usd=0.00024)


`asdict()` transforma a árvore inteira em dicts — pronto para mandar a um backend de observabilidade como JSON:

In [6]:
print(f"json.dumps(asdict(tracer.root), default=str)[:300]: {json.dumps(asdict(tracer.root), default=str)[:300]} ...")

json.dumps(asdict(tracer.root), default=str)[:300]: {"name": "support_agent", "kind": "agent", "start": 227.025618349, "end": 227.035803889, "attrs": {"user": "ana"}, "usage": {"input_tokens": 0, "output_tokens": 0, "calls": 0, "cost_usd": 0.0}, "children": [{"name": "plan", "kind": "llm", "start": 227.025625656, "end": 227.025645694, "attrs": {"mode ...


## 5. Harness de avaliação (evals)

Toda mudança de prompt, modelo ou schema deve passar por um conjunto de casos fixos. Casos e resultados são dataclasses
`frozen`: imutáveis, comparáveis, fáceis de serializar e versionar.

In [7]:
@dataclass(frozen=True, slots=True)
class EvalCase:
    id: str
    input: str
    expected: str
    tags: tuple[str, ...] = ()

@dataclass(frozen=True, slots=True)
class EvalResult:
    case: EvalCase
    predicted: str
    latency_ms: float

    @property
    def correct(self) -> bool:
        return self.predicted == self.case.expected

@dataclass(frozen=True)
class EvalReport:
    results: tuple[EvalResult, ...]

    @property
    def accuracy(self) -> float:
        return sum(r.correct for r in self.results) / len(self.results)

    def by_tag(self) -> dict[str, float]:
        acc: dict[str, list[bool]] = {}
        for r in self.results:
            for t in r.case.tags:
                acc.setdefault(t, []).append(r.correct)
        return {t: sum(v) / len(v) for t, v in acc.items()}

    def confusion(self) -> Counter:
        return Counter((r.case.expected, r.predicted) for r in self.results if not r.correct)

    def p95_latency(self) -> float:
        return statistics.quantiles([r.latency_ms for r in self.results], n=20)[-1]

CASES = (
    EvalCase("c1", "minha fatura veio errada", "billing", ("pt", "easy")),
    EvalCase("c2", "app dá erro ao abrir", "tech_support", ("pt", "easy")),
    EvalCase("c3", "quero cancelar o plano", "cancellation", ("pt", "easy")),
    EvalCase("c4", "cobraram 2x e o app trava", "billing", ("pt", "hard", "multi")),
    EvalCase("c5", "how much is the pro plan?", "sales", ("en",)),
    EvalCase("c6", "vocês têm plano anual?", "sales", ("pt",)),
    EvalCase("c7", "erro na fatura do mês", "billing", ("pt", "hard")),
    EvalCase("c8", "please cancel my account", "cancellation", ("en",)),
)

def run_eval(classify, cases: tuple[EvalCase, ...]) -> EvalReport:
    results = []
    for c in cases:
        t0 = time.perf_counter()
        pred = classify(c.input)
        results.append(EvalResult(c, pred, (time.perf_counter() - t0) * 1000))
    return EvalReport(tuple(results))

# "versão 1" do classificador: palavras-chave só em português
v1 = keyword_llm({"fatura": "billing", "cobra": "billing", "erro": "tech_support", "cancel": "cancellation",
                  "plano": "sales"}, default="other")
# "versão 2": prioriza cobrança e entende inglês
v2 = keyword_llm({"fatura": "billing", "cobra": "billing", "cancel": "cancellation", "plan": "sales",
                  "erro": "tech_support", "trava": "tech_support"}, default="other")

for name, clf in [("v1", v1), ("v2", v2)]:
    rep = run_eval(lambda text: clf([{"role": "user", "content": text}]), CASES)
    print(f"{name}: acurácia {rep.accuracy:.0%} | por tag {({k: f'{v:.0%}' for k, v in rep.by_tag().items()})}")
    print(f"    erros (esperado, previsto): {dict(rep.confusion())}")

v1: acurácia 88% | por tag {'pt': '100%', 'easy': '100%', 'hard': '100%', 'multi': '100%', 'en': '50%'}
    erros (esperado, previsto): {('sales', 'other'): 1}
v2: acurácia 100% | por tag {'pt': '100%', 'easy': '100%', 'hard': '100%', 'multi': '100%', 'en': '100%'}
    erros (esperado, previsto): {}


O relatório mostra **onde** a v1 erra — a tag `en` cai para 50%: `"pro plan"` não casa com a palavra-chave `"plano"` — e confirma
que a v2 corrige sem regredir os outros casos. É exatamente o tipo de evidência que deve acompanhar uma mudança de prompt.
Num eval real, o `classify` chamaria o LLM e o `EvalReport` seria salvo (via `asdict`) para comparar versões ao longo do tempo.

## 6. Por que dataclass aqui (e não Pydantic)?

| critério | runtime interno do agente |
|---|---|
| origem do dado | o próprio código (confiável) |
| volume | milhares de objetos por execução (mensagens, spans) |
| precisa converter tipos? | não |
| precisa de JSON Schema? | não (ninguém de fora preenche) |
| precisa ser imutável/hashable? | sim, frequentemente (`LLMConfig` como chave de cache, `EvalCase`) |
| dependências | stdlib — o núcleo pode ser reutilizado em workers, CLIs e testes |

Quando um desses objetos **cruzar a borda** (ex.: o `EvalReport` virar resposta de uma API), aí sim um modelo Pydantic de saída
— ou `TypeAdapter(EvalReport)` — entra em cena (veja `dataclass/03`, seção 9).

Como fica, na prática, levar o `EvalReport` para a borda — as duas opções:

In [8]:
from pydantic import BaseModel, TypeAdapter

# Opção 1: TypeAdapter serializa a dataclass como ela é (campos, sem as @property)
ReportAdapter = TypeAdapter(EvalReport)
print(f"ReportAdapter.dump_json(rep).decode()[:160]: {ReportAdapter.dump_json(rep).decode()[:160]} ...")

# Opção 2: modelo Pydantic de saída com o contrato que a API expõe (inclui métricas calculadas)
class EvalReportOut(BaseModel):
    cases: int
    accuracy: float
    accuracy_by_tag: dict[str, float]
    errors: list[tuple[str, str]]

    @classmethod
    def from_report(cls, r: EvalReport) -> "EvalReportOut":
        return cls(cases=len(r.results), accuracy=r.accuracy, accuracy_by_tag=r.by_tag(), errors=list(r.confusion()))

print(f"EvalReportOut.from_report(rep).model_dump_json(): {EvalReportOut.from_report(rep).model_dump_json()}")

ReportAdapter.dump_json(rep).decode()[:160]: {"results":[{"case":{"id":"c1","input":"minha fatura veio errada","expected":"billing","tags":["pt","easy"]},"predicted":"billing","latency_ms":0.00131400000213 ...
EvalReportOut.from_report(rep).model_dump_json(): {"cases":8,"accuracy":1.0,"accuracy_by_tag":{"pt":1.0,"easy":1.0,"hard":1.0,"multi":1.0,"en":1.0},"errors":[]}


➡️ Próximo: [`05_agent_pipeline_end_to_end.ipynb`](05_agent_pipeline_end_to_end.ipynb)